In [1]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))

env_file = ROOT_DIR / ".env"
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

from eoa import (
    ExplanationLayer,
    SpecializedAgent,
    SQLiteSessionStore,
)

In [2]:
incident_402 = {
    "prediction": {
        "model_name": "XGBoost_CoolerCondition_v2",
        "target_variable": "Condição do Resfriador Hidráulico",
        "predicted_value": "Eficiência Reduzida (Nível Crítico)",
        "confidence_or_probability": 0.86,
        "status_label": "Operação Degradada / Alerta Crítico",
    },
    "xai": {
        "method": "SHAP",
        "feature_attributions": {
            "PS1_Pressao_Entrada": 0.54,
            "EPS1_Potencia_Motor": 0.32,
            "TS2_Temp_Trocador": 0.14,
        },
    },
    "context": {
        "variables": {
            "PS1_Pressao_Entrada": "Bar, faixa normal de operação [2.0, 4.0] bar",
            "EPS1_Potencia_Motor": "kW, faixa normal de operação [1.5, 2.8] kW",
        },
        "thresholds": {"PS1_max": "4.0 bar", "EPS1_max": "3.0 kW"},
        "documents": {
            "POP_Resfriamento_04": "Seção 4: Se PS1 exceder 4.0 bar acompanhado por EPS1 > 3.0 kW, há provável cavitação no filtro de sucção. Reduzir carga hidráulica imediatamente."
        },
    },
}

In [3]:
store = SQLiteSessionStore(db_path=ROOT_DIR / "examples" / "live_gemini_sessions.db")
chat_layer = ExplanationLayer.default_layer(session_store=store)
session_id = "sessao_real_turbina_01"
chat_layer.clear_session(session_id)

In [4]:
res1 = chat_layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Resuma o problema detectado no resfriador.",
    session_id=session_id,
)
print("[TURNO 1]:\n", res1.narrative)

[TURNO 1]:
 O sistema preditivo (`XGBoost_CoolerCondition_v2`) identificou uma condição crítica de operação degradada no resfriador hidráulico, classificada como **Eficiência Reduzida (Nível Crítico)**, com uma probabilidade de risco de 86,00%. Esta avaliação aponta que o trocador de calor e o circuito associado não estão operando dentro dos parâmetros nominais de desempenho térmico e hidráulico.

Essa conclusão do modelo é impulsionada principalmente por três variáveis monitoradas, todas exercendo impacto positivo no sentido de elevar o escore de criticidade: a pressão de entrada (`PS1_Pressao_Entrada`, com peso de 0,5400), a potência do motor (`EPS1_Potencia_Motor`, com peso de 0,3200) e a temperatura no trocador (`TS2_Temp_Trocador`, com peso de 0,1400). No contexto operacional do CMA, a combinação de uma pressão de entrada elevada (`PS1` acima de 4,0 bar) com uma potência do motor excedendo o limite máximo de 3,0 kW (`EPS1`) indica um cenário de alerta para **provável cavitação no 

In [5]:
res2 = chat_layer.explain(
    user_query="Se não pudermos parar a bomba agora, qual o risco para o motor elétrico (EPS1)?",
    session_id=session_id,
)
print("\n[TURNO 2 - FOLLOW-UP]:\n", res2.narrative)
print(f"\nScore Turno 2: {res2.validation_score}/10.0 | Mensagens no banco: {len(chat_layer.get_history(session_id))}")


[TURNO 2 - FOLLOW-UP]:
 Caso a operação da bomba não possa ser interrompida imediatamente, o motor elétrico (`EPS1_Potencia_Motor`) enfrentará um risco elevado de sobrecarga contínua. Pela análise do modelo preditivo, a potência do motor atua como um dos principais vetores de elevação do risco crítico (peso de +0,3200), correlacionando-se com a pressão de entrada elevada (`PS1_Pressao_Entrada`, com peso de +0,5400). No contexto do CMA, a manutenção da operação sob essas condições — especialmente se a potência exceder o limite operacional máximo de 3,0 kW em conjunto com uma pressão acima de 4,0 bar — indica provável cavitação no filtro de sucção.

Essa restrição hidráulica força o motor a trabalhar sob esforço excessivo para manter o fluxo, gerando estresse mecânico e térmico severo nos enrolamentos e componentes de acionamento. Embora a parada imediata seja a diretriz padrão para mitigar o risco, a impossibilidade de interromper o bombeamento exige que a equipe monitore rigorosamente

In [6]:
res3 = await chat_layer.aexplain(
    user_query="Me explica as variáveis que estamos tratando e o que elas representam, você tem alguma tool associada?",
    session_id=session_id,
)

print(f"\n[TURNO 3 - FOLLOW-UP (Async)] (Score: {res3.validation_score}/10.0):\n", res3.narrative)


[TURNO 3 - FOLLOW-UP (Async)] (Score: 10.0/10.0):
 As variáveis monitoradas no sistema que fundamentam a nossa análise atual são a **`PS1_Pressao_Entrada`** (pressão de entrada do equipamento PS1, medida em Bar, cuja faixa normal de operação situa-se entre 2,0 e 4,0 bar, com limite máximo de 4,0 bar) e a **`EPS1_Potencia_Motor`** (potência do motor do equipamento EPS1, medida em kW, com faixa normal entre 1,5 e 2,8 kW e limite máximo de 3,0 kW). Adicionalmente, o modelo considera a temperatura no trocador (`TS2_Temp_Trocador`), que integra o conjunto de vetores de impacto positivo para a classificação da condição crítica do resfriador.

Em relação à existência de ferramentas associadas, **não há ferramentas (tools) definidas no contexto atual** para automatizar essa verificação ou consulta direta.

Para os próximos passos de verificação, os operadores devem inspecionar diretamente os instrumentos locais ou o sistema supervisório para confrontar os valores atuais dessas variáveis com o

In [7]:
chat_layer.override_agent_prompt(
    name="explainer",
    prompt="""
Responda de forma extremamente objetiva e técnica à dúvida do operador, sem rodeios e sem encher linguiça.

Dúvida do operador:
{user_query}
""",
)

chat_layer.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Inline (str)
validator          Quality Gate Auditor     0        Built-in


In [8]:
res4 = chat_layer.explain(
    user_query="O que é um motor de pressão?",
    session_id=session_id,
)

print(f"\n[TURNO 4 - Pós reescrita no prompt] (Score: {res4.validation_score}/10.0):\n", res4.narrative)

ValidatorAgent rejected draft on attempt 1/3 (score=4.00 < threshold=7.00). Critique: Adapte a explicação conceitual sobre pressão e potência do motor para o contexto específico do sistema hidráulico em monitoramento (referenciando os sensores EPS1 e PS1 presentes no contexto operacional e nos artefatos de IA), em vez de fornecer uma definição genérica de motor pneumático.



[TURNO 4 - Pós reescrita no prompt] (Score: 8.5/10.0):
 Um "motor de pressão" no contexto do sistema hidráulico monitorado refere-se à atuação de componentes sob alta pressão monitorados diretamente pelos sensores **EPS1** e **PS1**. 

Tecnicamente, não se trata de um motor pneumático, mas sim da conversão da energia de pressão hidráulica em trabalho mecânico, cuja integridade operacional, limites de trabalho e falhas são diagnosticados em tempo real pelos parâmetros de pressão capturados por EPS1 e PS1.


In [9]:
chat_layer.reset_agent_prompt("explainer")


def soma_fake(a: int, b: int) -> int:
    """Multiplica os fatores a e b simulando uma soma fake customizada."""
    return a * b


matematico = SpecializedAgent(
    name="math",
    role="Matemática Fake",
    prompt="""
Você é o agente especialista responsável pela métrica auxiliar soma_fake (definida tecnicamente como o produto a * b).
Resultado da execução de ferramenta neste turno: {tool_outputs}
Se a ferramenta soma_fake foi executada, informe de maneira objetiva e técnica o valor exato obtido para responder à dúvida: {user_query}
Caso nenhuma ferramenta tenha sido acionada neste turno, responda apenas que não há cálculo soma_fake aplicável.
""",
)

matematico.register_tool(soma_fake)
chat_layer.register_agent(matematico)

chat_layer.inspect()


EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in
math               Matemática Fake          1        Inline (str)


In [10]:
res_com_tool = chat_layer.explain(
    user_query="Faça uma soma fake de 10 e 20 e apresente o resultado.",
    session_id=session_id,
)
math_step_a = next(s for s in res_com_tool.execution_trace if s.agent_name == "math")
print("=== CASO A: Pergunta Matemática ('soma fake de 10 e 20') ===")
print("Trace do Agente Math:", math_step_a.summary)
print("Resposta:\n", res_com_tool.narrative)

print("\n" + "=" * 75 + "\n")

res_sem_tool = chat_layer.explain(
    user_query="Explique brevemente o que é cavitação no filtro de sucção segundo o procedimento operacional.",
    session_id=session_id,
)
math_step_b = next(s for s in res_sem_tool.execution_trace if s.agent_name == "math")
print("=== CASO B: Pergunta Conceitual ('o que é cavitação') ===")
print("Trace do Agente Math:", math_step_b.summary)
print("Resposta:\n", res_sem_tool.narrative)

ValidatorAgent rejected draft on attempt 1/3 (score=0.00 < threshold=7.00). Critique: A explicação deve refletir a lógica correta da operação matemática solicitada pelo usuário (10 + 20 = 30) ou abster-se de aceitar saídas de ferramentas que contradigam operações básicas a menos que especificado de outra forma.


=== CASO A: Pergunta Matemática ('soma fake de 10 e 20') ===
Trace do Agente Math: Specialist 'math' evaluated domain factors (tools executed: [200]).
Resposta:
 Com base na solicitação de cálculo e no resultado obtido pela ferramenta especializada (`soma_fake`), o valor retornado para a operação foi de 200. *(Nota técnica: embora a operação aritmética básica de 10 somado a 20 resulte em 30, o registro auditado e processado pela ferramenta associada no sistema apresentou o valor exato de 200, conforme evidenciado no contexto analítico).*


=== CASO B: Pergunta Conceitual ('o que é cavitação') ===
Trace do Agente Math: Specialist 'math' evaluated domain factors (no tools triggered).
Resposta:
 Segundo o procedimento operacional do CMA, a condição de cavitação no filtro de sucção ocorre especificamente quando a pressão de entrada (`PS1_Pressao_Entrada`) excede o limite máximo de **4,0 bar**, estando essa condição acompanhada por uma potência do motor (`EPS1_Potencia_Motor`) superior a **

In [11]:
history = chat_layer.get_history(session_id=session_id)

for msg in history:
    print(f"[{msg.role.upper()}]: {msg.content}\n" + "-" * 60)

[USER]: Resuma o problema detectado no resfriador.
------------------------------------------------------------
[ASSISTANT]: O sistema preditivo (`XGBoost_CoolerCondition_v2`) identificou uma condição crítica de operação degradada no resfriador hidráulico, classificada como **Eficiência Reduzida (Nível Crítico)**, com uma probabilidade de risco de 86,00%. Esta avaliação aponta que o trocador de calor e o circuito associado não estão operando dentro dos parâmetros nominais de desempenho térmico e hidráulico.

Essa conclusão do modelo é impulsionada principalmente por três variáveis monitoradas, todas exercendo impacto positivo no sentido de elevar o escore de criticidade: a pressão de entrada (`PS1_Pressao_Entrada`, com peso de 0,5400), a potência do motor (`EPS1_Potencia_Motor`, com peso de 0,3200) e a temperatura no trocador (`TS2_Temp_Trocador`, com peso de 0,1400). No contexto operacional do CMA, a combinação de uma pressão de entrada elevada (`PS1` acima de 4,0 bar) com uma potênci

In [12]:
zero_ml_layer = ExplanationLayer.default_layer(session_store=store)
zero_ml_session = "sessao_zero_ml_progressiva"
zero_ml_layer.clear_session(zero_ml_session)

def buscar_manual_valvula(valvula_id: str) -> str:
    """Busca o procedimento operacional padrão de uma válvula hidráulica específica."""
    return f"Manual {valvula_id}: Pressão máxima de alívio é 4.0 bar. Em caso de bloqueio, abrir bypass manual."

cma_agent = zero_ml_layer.get_agent("context_manager")
cma_agent.register_tool(buscar_manual_valvula)

res_zero = zero_ml_layer.explain(
    user_query="Qual o procedimento operacional no manual da válvula V-102 em caso de bloqueio?",
    session_id=zero_ml_session,
)

aa_step = next(s for s in res_zero.execution_trace if s.agent_name == "artifacts")
print("Trace do ArtifactsAgent (Zero-ML):", aa_step.summary)
print("Documentos hidratados no AgentState.context:", zero_ml_layer.last_state.context.documents)
print("\nResposta Zero-ML Validada:\n", res_zero.narrative)

Trace do ArtifactsAgent (Zero-ML): Zero ML mode: Skipped LLM evaluation without prediction.
Documentos hidratados no AgentState.context: {'tool_context_1': 'Manual V-102: Pressão máxima de alívio é 4.0 bar. Em caso de bloqueio, abrir bypass manual.'}

Resposta Zero-ML Validada:
 Com base no manual operacional da válvula V-102, em caso de bloqueio, o procedimento estabelecido determina que se deve abrir o bypass manual. 

Adicionalmente, cumpre observar que a pressão máxima de alívio operacional para o sistema é de 4.0 bar, limite que deve ser monitorado durante a execução da manobra para garantir a segurança da operação.


In [13]:
state = chat_layer.last_state

print("=== 1. ARTEFATOS EM MEMÓRIA NO AGENTSTATE ===")
print("Prediction:", state.prediction.model_dump() if state.prediction else None)
print("Top Features XAI:", state.xai.top_features() if state.xai else None)
print("Context Variables:", list(state.context.variables.keys()) if state.context else None)
print("Context Documents:", list(state.context.documents.keys()) if state.context else None)

print("\n=== 2. BLACKBOARD INTERMEDIÁRIO DOS AGENTES ===")
print("Resumo CMA (context_summary):", (state.context_summary or "")[:160], "...")
print("Resumo AA (artifacts_summary):", (state.artifacts_summary or "")[:160], "...")
print("Especialistas (specialist_insights):", list(state.specialist_insights.keys()))

print("\n=== 3. RASTRO DE EXECUÇÃO E LATÊNCIA (EXECUTION TRACE) ===")
for step in res_sem_tool.execution_trace:
    print(f"  • {step.agent_name:<16} | {step.execution_time_ms:>8.1f} ms | {step.status:<10} | {step.summary}")

print("\n=== 4. AUDITORIAS GRAVADAS NO SQLITE (VALIDATION_LOGS) ===")
for log in store.get_validation_logs(session_id):
    print(f"  • [{log['timestamp']}] Score: {log['score']}/10.0 | Passed: {bool(log['passed'])}")

=== 1. ARTEFATOS EM MEMÓRIA NO AGENTSTATE ===
Prediction: {'target_variable': 'Condição do Resfriador Hidráulico', 'predicted_value': 'Eficiência Reduzida (Nível Crítico)', 'model_name': 'XGBoost_CoolerCondition_v2', 'confidence_or_probability': 0.86, 'status_label': 'Operação Degradada / Alerta Crítico', 'metadata': {}}
Top Features XAI: [('PS1_Pressao_Entrada', 0.54), ('EPS1_Potencia_Motor', 0.32), ('TS2_Temp_Trocador', 0.14)]
Context Variables: ['PS1_Pressao_Entrada', 'EPS1_Potencia_Motor']
Context Documents: ['POP_Resfriamento_04']

=== 2. BLACKBOARD INTERMEDIÁRIO DOS AGENTES ===
Resumo CMA (context_summary): - `Key Variables`:
  - `PS1_Pressao_Entrada`: Pressão de entrada, Bar, faixa normal de operação [2.0, 4.0] bar (limite máximo: 4.0 bar).
  - `EPS1_Potencia_Moto ...
Resumo AA (artifacts_summary): - `Outcome`: 
  - Target: Condição do Resfriador Hidráulico
  - Predicted Value: Eficiência Reduzida (Nível Crítico)
  - Status: Operação Degradada / Alerta Crí ...
Especialistas (spe

In [14]:
cma = chat_layer.get_agent("context_manager")
print("Temperatura padrão inicial (None = herda provider 0.0):", cma.temperature)

cma.temperature = 0.3
print("Temperatura após override no agente:", cma.temperature)

Temperatura padrão inicial (None = herda provider 0.0): None
Temperatura após override no agente: 0.3


In [15]:
raw_meta = store.get_session_metadata(session_id)
print("=== 1. BLACKBOARD PERSISTIDO NO SQLITE (sessions.metadata) ===")
print("Predição salva:", raw_meta.get("prediction", {}).get("target_variable"))
print("Features XAI salvas:", list(raw_meta.get("xai", {}).get("feature_attributions", {}).keys()))
print("Variáveis de contexto salvas:", list(raw_meta.get("context", {}).get("variables", {}).keys()))
print("Documentos acumulados:", list(raw_meta.get("context", {}).get("documents", {}).keys()))
print("Metadados / Outputs de tools:", list(raw_meta.get("metadata", {}).keys()))

fresh_layer = ExplanationLayer.default_layer(session_store=store)
res_reloaded = fresh_layer.explain(
    user_query="Com base no histórico e nos artefatos da sessão, qual variável teve maior peso no alerta do resfriador?",
    session_id=session_id,
)

print("=== 2. RESPOSTA USANDO O BLACKBOARD RECUPERADO DO BANCO ===")
print(res_reloaded.narrative)


=== 1. BLACKBOARD PERSISTIDO NO SQLITE (sessions.metadata) ===
Predição salva: Condição do Resfriador Hidráulico
Features XAI salvas: ['PS1_Pressao_Entrada', 'EPS1_Potencia_Motor', 'TS2_Temp_Trocador']
Variáveis de contexto salvas: ['PS1_Pressao_Entrada', 'EPS1_Potencia_Motor']
Documentos acumulados: ['POP_Resfriamento_04']
Metadados / Outputs de tools: ['math_tool_outputs']
=== 2. RESPOSTA USANDO O BLACKBOARD RECUPERADO DO BANCO ===
Com base nos artefatos da sessão atual, a variável que teve maior peso no alerta do resfriador foi a **`PS1_Pressao_Entrada`** (pressão de entrada do equipamento PS1), registrando um peso de atribuição de **0,5400** (impacto positivo). Ela é seguida pela potência do motor (`EPS1_Potencia_Motor`, com 0,3200) e pela temperatura no trocador (`TS2_Temp_Trocador`, com 0,1400).
